In [2]:
import numpy as np
import h5py
from tqdm import tqdm

def process_data_and_get_corr_matrix(num_pairs, v1, v2):
    """
    Process data and return the correlation matrices without leave-one-out iteration.
    """
    corr_matrices = []

    for pair_num in range(num_pairs):
        send_stack = v1[pair_num]
        rece_stack = v2[pair_num]
        
        # Concatenate the two conditions
        pair_matrix = np.concatenate((send_stack, rece_stack), axis=0)

        # Calculate the correlation matrix
        corr_matrix = np.corrcoef(pair_matrix)

        # Fisher's Z Transformation
        fisher_z_matrix = 0.5 * np.log((1 + corr_matrix) / (1 - corr_matrix))

        # Extract the part of the matrix that correlates the two conditions
        corr_matrix_half = fisher_z_matrix[:send_stack.shape[0], send_stack.shape[0]:]  
        
        corr_matrices.append(corr_matrix_half)

    corr_matrices = np.array(corr_matrices)
    return corr_matrices

def calculate_z_corr_matrix2(corr_matrices_12, corr_matrices_34):
    corr_matrices_34 = np.array(corr_matrices_34)
    corr_matrices_12 = np.array(corr_matrices_12)
    corr_matrix_all = np.concatenate((corr_matrices_12, corr_matrices_34), axis=1)
    return corr_matrix_all

def sliding_window_analysis(data_A, data_B, window_size=2, step_size=1):
    """
    data_A: 时间序列数据 (23被试, 12试次, time_points, voxel)
    data_B: 目标数据 (23被试, 12试次, voxel)
    """

    n_timepoints = data_A.shape[2]
    n_windows = (n_timepoints - window_size + 1) // step_size
    
    all_correlations = []
    
    for w in range(n_windows):
        start_idx = w * step_size
        end_idx = start_idx + window_size
        
        window_mean = np.mean(data_A[..., start_idx:end_idx, :], axis=-2)
        
        corr = process_data_and_get_corr_matrix(23, window_mean, data_B)
        all_correlations.append(corr)
    
    return all_correlations

def load_data(file_path):
    """Function to load data from subarray_0 to subarray_399 from a given file path."""
    with h5py.File(file_path, 'r') as file:
        data_list = []
        for i in range(400):
            dataset_name = f'subarray_{i}'
            if dataset_name in file:
                data = file[dataset_name][:]
                data_list.append(data)
            else:
                print(f"Dataset '{dataset_name}' not found in the file.")
        return data_list


def load_split_data(file_path):
    """
    加载分成f2和f5两组的数据
    file_path: h5文件路径
    返回: f2_list, f5_list, 每个list包含400个脑区的数据
    """
    f2_list = []
    f5_list = []
    
    with h5py.File(file_path, 'r') as file:
        # 加载f2组数据
        f2_group = file['f2']
        # 加载f5组数据
        f5_group = file['f5']
        
        # 获取所有脑区的数据
        for i in range(400):
            region_name = f'region_{i}'
            if region_name in f2_group and region_name in f5_group:
                f2_data = f2_group[region_name][:]
                f5_data = f5_group[region_name][:]
                f2_list.append(f2_data)
                f5_list.append(f5_data)
            else:
                print(f"Region {i} not found in the file.")
                
    return f2_list, f5_list


def check_for_nans(data_list, list_name):
    """Function to check for NaN values in a list of datasets."""
    for i, dataset in enumerate(data_list):
        if np.isnan(dataset).any():
            print(f"NaN found in dataset {i} of {list_name}")

def main():
    # 定义文件路径
    file_path1 = 'face10156_speak_va.h5'
    file_path2 = 'face10156_listen_vb_noavg.h5'
    file_path3 = 'face10156_speak_vb.h5'
    file_path4 = 'face10156_listen_va_noavg.h5'

    # 加载数据
    print("Loading data...")
    # 加载普通格式数据
    stacked_vas_list = load_data(file_path1)
    stacked_vbs_list = load_data(file_path3)
    
    # 加载分组数据
    vbr_f2_list, vbr_f5_list = load_split_data(file_path2)
    var_f2_list, var_f5_list = load_split_data(file_path4)

    all_z_correlations_real = []

    # 主分析循环
    for brain_area in tqdm(range(400), desc="Processing brain areas"):
        # 获取当前脑区的数据
        vas_data = stacked_vas_list[brain_area]
        vbs_data = stacked_vbs_list[brain_area]
        
        # f2部分的相关分析
        real_corr_rbsa_f2 = sliding_window_analysis(vbr_f2_list[brain_area], vas_data[:, :12, :])
        real_corr_rasb_f2 = sliding_window_analysis(var_f2_list[brain_area], vbs_data[:, :12, :])
        
        # f5部分的相关分析
        real_corr_rbsa_f5 = sliding_window_analysis(vbr_f5_list[brain_area], vas_data[:, 12:, :])
        real_corr_rasb_f5 = sliding_window_analysis(var_f5_list[brain_area], vbs_data[:, 12:, :])
        
        # 合并f2和f5的结果
        real_corr_rbsa = real_corr_rbsa_f2 + real_corr_rbsa_f5
        print(np.array(real_corr_rbsa).shape)

        real_corr_rasb = real_corr_rasb_f2 + real_corr_rasb_f5

        # 计算Z相关矩阵
        z_corr_matrix_real = calculate_z_corr_matrix2(real_corr_rbsa, real_corr_rasb)
        all_z_correlations_real.append(z_corr_matrix_real)

    # 转换为numpy数组和保存结果
    results = {
        'z_correlations_real': np.array(all_z_correlations_real)
    }

    print("Saving results...")
    with h5py.File('slidewindow2_realcorrelation_slface.h5', 'w') as f:
        for key, value in results.items():
            f.create_dataset(key, data=value)

    print("Analysis completed!")
    return results

if __name__ == "__main__":
    results = main()

Loading data...


Processing brain areas:   0%|                                                                   | 0/400 [00:00<?, ?it/s]/tmp/ipykernel_642194/1488242382.py:22: RuntimeWarning: divide by zero encountered in divide
  fisher_z_matrix = 0.5 * np.log((1 + corr_matrix) / (1 - corr_matrix))
Processing brain areas:   0%|▎                                                          | 2/400 [00:00<00:28, 13.88it/s]

(23, 23, 12, 12)
(23, 23, 12, 12)
(23, 23, 12, 12)


Processing brain areas:   2%|▉                                                          | 6/400 [00:00<00:29, 13.53it/s]

(23, 23, 12, 12)
(23, 23, 12, 12)
(23, 23, 12, 12)


...

Processing brain areas:  99%|████████████████████████████████████████████████████████▎| 395/400 [00:21<00:00, 17.89it/s]

(23, 23, 12, 12)
(23, 23, 12, 12)
(23, 23, 12, 12)
(23, 23, 12, 12)


Processing brain areas: 100%|█████████████████████████████████████████████████████████| 400/400 [00:21<00:00, 18.36it/s]

(23, 23, 12, 12)
(23, 23, 12, 12)
(23, 23, 12, 12)
(23, 23, 12, 12)


Saving results...
Analysis completed!


In [9]:
import h5py
import numpy as np

# 加载保存的结果文件
with h5py.File('slidewindow_correlation_results_window3.h5', 'r') as f:
    # 加载z相关矩阵
    z_correlations_real = f['z_correlations_real'][:]

# 打印形状
print("z_correlations_real shape:", z_correlations_real.shape)


z_correlations_real shape: (400, 19, 46, 12, 12)


In [1]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm # 用于显示被试处理进度

def calculate_embedding_rdm(feature_vectors):
    """
    根据特征向量计算 RDM (trial-by-trial)。
    
    参数:
    feature_vectors (np.array): 形状为 (n_trials, n_features)
    
    返回:
    np.array: 形状为 (n_trials, n_trials) 的RDM
    """
    n_trials, n_features = feature_vectors.shape
    
    if n_trials < 2:
        print("警告: 试次数少于2，无法计算RDM。")
        return np.array([])
        
    # np.corrcoef 期望 (variables, observations) 或 (observations, variables)
    # 输入 (n_trials, n_features)，它会计算 n_trials 两两之间的相关性
    # 得到 (n_trials, n_trials) 矩阵
    corr_matrix = np.corrcoef(feature_vectors)
    
    # 处理NaN (例如，如果某个trial的特征向量方差为0)
    # 我们将NaN的相关性设为0
    corr_matrix = np.nan_to_num(corr_matrix, nan=0.0)
    
    # RDM = 1 - Correlation
    rdm = 1.0 - corr_matrix
    
    return rdm

# --- 主执行流程 ---

# 1. 定义输入和输出
# !!! 请将 'your_embedding_file.csv' 替换为您的实际文件名 !!!
EMBEDDING_FILE = '/mnt/e/2022_hyper_audiorecord/trial_embeddings_all_subjects_para_trialtext.csv' 
OUTPUT_DIR = 'embedding_rdms'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"正在从 {EMBEDDING_FILE} 加载 embedding 数据...")

# 2. 加载数据
try:
    # *** 主要修改点: 默认使用 sep=',' ***
    data = pd.read_csv(EMBEDDING_FILE, sep=',')
    
    # 健壮性检查：如果加载后只有一列，尝试用制表符分隔
    if data.shape[1] == 1:
        print("检测到单列，尝试使用制表符 (TSV) 分隔符...")
        data = pd.read_csv(EMBEDDING_FILE, sep='\t')
        
    # 检查必需的列
    required_cols = ['subject_id', 'condition', 'condition_trial_num']
    if not all(col in data.columns for col in required_cols):
        print(f"错误: 文件中缺少必需的列。需要: {required_cols}")
        print(f"找到的列: {data.columns.tolist()}")
        # 在实际使用中，您可能希望在这里退出
        # exit() 

except FileNotFoundError:
    print(f"错误: 文件未找到 {EMBEDDING_FILE}")
    # exit()
except Exception as e:
    print(f"加载文件时出错: {e}")
    # exit()

if 'data' in locals(): # 确保 'data' 已成功加载
    print("数据加载成功。")
    print(f"文件包含 {data.shape[1] - 3} 个特征维度。")

    # 3. 获取所有唯一的被试ID
    subject_ids = data['subject_id'].unique()
    subject_ids.sort() # 排序
    
    print(f"共找到 {len(subject_ids)} 个被试。开始处理...")

    # 4. 逐个被试循环
    for sub_id in tqdm(subject_ids, desc="Processing Subjects"):
        
        # 筛选当前被试的数据
        sub_data = data[data['subject_id'] == sub_id].copy()
        
        # --- 4.1 处理 f2 ---
        f2_data = sub_data[sub_data['condition'] == 'f2']
        
        if not f2_data.empty:
            # 关键：按 trial 编号排序，确保RDM的行/列顺序正确
            f2_data = f2_data.sort_values(by='condition_trial_num')
            
            # 提取特征：从第4列 (索引3) 到最后一列
            f2_features = f2_data.iloc[:, 3:].values 
            
            # 计算 RDM
            f2_rdm = calculate_embedding_rdm(f2_features)
            
            # 定义保存路径
            f2_filename = os.path.join(OUTPUT_DIR, f'sub-{sub_id}_condition-f2_rdm.npy')
            np.save(f2_filename, f2_rdm)
        else:
            print(f"警告: 被试 {sub_id} 未找到 'f2' 数据。")
            
        # --- 4.2 处理 f5 ---
        f5_data = sub_data[sub_data['condition'] == 'f5']
        
        if not f5_data.empty:
            # 关键：按 trial 编号排序
            f5_data = f5_data.sort_values(by='condition_trial_num')
            
            # 提取特征
            f5_features = f5_data.iloc[:, 3:].values
            
            # 计算 RDM
            f5_rdm = calculate_embedding_rdm(f5_features)
            
            # 定义保存路径
            f5_filename = os.path.join(OUTPUT_DIR, f'sub-{sub_id}_condition-f5_rdm.npy')
            np.save(f5_filename, f5_rdm)
        else:
            print(f"警告: 被试 {sub_id} 未找到 'f5' 数据。")

    print("\n--- Embedding RDM 处理完成! ---")
    print(f"所有RDM文件已保存至 '{OUTPUT_DIR}' 目录。")

正在从 /mnt/e/2022_hyper_audiorecord/trial_embeddings_all_subjects_para_trialtext.csv 加载 embedding 数据...
数据加载成功。
文件包含 384 个特征维度。
共找到 29 个被试。开始处理...


Processing Subjects: 100%|█████████████████████████████████████████████████████████████| 29/29 [00:00<00:00, 905.28it/s]


--- Embedding RDM 处理完成! ---
所有RDM文件已保存至 'embedding_rdms' 目录。


--- 开始 *合并后* 的表征对齐分析 ---

--- 正在加载 *筛选后* 的 Embedding RDMs ---
按顺序加载 0 个被试的 Model RDMs:
按顺序加载 0 个被试的 Model RDMs:
错误：未能加载 Model RDMs 或 Model RDMs 为空。
请检查 'embedding_rdms_filtered' 目录中是否存在 RDM 文件。
Model RDMs 向量化完毕。形状 (f2): (0,)
Model RDMs 向量化完毕。形状 (f5): (0,)
---------------------------------
将使用 -2 个核心进行并行计算...

正在并行处理 combined_f2_rdms ...
  fMRI RDM 输入: rdm_results_combined/combined_f2_rdms
  对齐结果输出: alignment_results_combined/combined_f2_rdms_alignment


Aligning combined_f2_rdms: 100%|█████████████████████████████████████████████████| 400/400 [00:00<00:00, 610.14region/s]


--- combined_f2_rdms 对齐完成 ---
成功: 0 | 跳过(文件未找到): 0 | 失败: 400

正在并行处理 combined_f5_rdms ...
  fMRI RDM 输入: rdm_results_combined/combined_f5_rdms
  对齐结果输出: alignment_results_combined/combined_f5_rdms_alignment


Aligning combined_f5_rdms: 100%|████████████████████████████████████████████████| 400/400 [00:00<00:00, 6581.62region/s]


--- combined_f5_rdms 对齐完成 ---
成功: 0 | 跳过(文件未找到): 0 | 失败: 400

--- 所有 *合并后* 的表征对齐分析已全部完成 ---
所有结果均已保存至 'alignment_results_combined' 目录中。

--- ERROR: 处理 combined_f2_rdms 脑区 8 时出错: 被试数量不匹配! fMRI有 29 个, 而Embedding有 0 个。 ---

--- ERROR: 处理 combined_f2_rdms 脑区 72 时出错: 被试数量不匹配! fMRI有 29 个, 而Embedding有 0 个。 ---

--- ERROR: 处理 combined_f2_rdms 脑区 77 时出错: 被试数量不匹配! fMRI有 29 个, 而Embedding有 0 个。 ---

--- ERROR: 处理 combined_f2_rdms 脑区 82 时出错: 被试数量不匹配! fMRI有 29 个, 而Embedding有 0 个。 ---

--- ERROR: 处理 combined_f2_rdms 脑区 88 时出错: 被试数量不匹配! fMRI有 29 个, 而Embedding有 0 个。 ---

--- ERROR: 处理 combined_f2_rdms 脑区 93 时出错: 被试数量不匹配! fMRI有 29 个, 而Embedding有 0 个。 ---

--- ERROR: 处理 combined_f2_rdms 脑区 101 时出错: 被试数量不匹配! fMRI有 29 个, 而Embedding有 0 个。 ---

--- ERROR: 处理 combined_f2_rdms 脑区 118 时出错: 被试数量不匹配! fMRI有 29 个, 而Embedding有 0 个。 ---

--- ERROR: 处理 combined_f2_rdms 脑区 140 时出错: 被试数量不匹配! fMRI有 29 个, 而Embedding有 0 个。 ---

--- ERROR: 处理 combined_f2_rdms 脑区 141 时出错: 被试数量不匹配! fMRI有 29 个, 而Embedding有 0 个。 ---

--- ERROR: